<a href="https://colab.research.google.com/github/victorDomingosSouza/analise-cancelamento-telecom/blob/main/Analise_de_cancelamentos_Telecom.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Analise de cancelamentos Telecom**

In [ ]:
import pandas as pd
import plotly.express as px

In [ ]:
Caminho = '/content/Telco_customer_churn - Telco_Churn.csv'
df = pd.read_csv(Caminho)
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce').fillna(0)

Taxa de cancelamento

In [ ]:
total_clientes = len(df)
total_cancelados = (df['Churn Label'] == 'Yes').sum()
taxa_cancelamento = (total_cancelados / total_clientes) * 100

print(f"Total de Clientes: {total_clientes:,}".replace(',', '.'))
print(f"Total de Cancelamentos: {total_cancelados:,}".replace(',', '.'))
print(f"Taxa Global de Cancelamento: {taxa_cancelamento:.2f}%\n")

ANÁLISE DE CONTRATOS

In [ ]:
fig_status = px.pie(
    df,
    names='Churn Label',
    title='<b>Proporção Geral de Clientes por Status</b>',
    hole=0.4,
    color='Churn Label',
    color_discrete_map={'No': '#2ecc71', 'Yes': '#e74c3c'}
)
fig_status.update_layout(template='plotly_white')
fig_status.show()

# Gráfico de Barras Empilhadas - Churn por Contrato
cruzamento_pct = pd.crosstab(df['Contract'], df['Churn Label'], normalize='index') * 100
df_contract_plot = cruzamento_pct.reset_index().melt(
    id_vars='Contract',
    var_name='Churn Label',
    value_name='Porcentagem'
)

fig_contract = px.bar(
    df_contract_plot,
    x='Contract',
    y='Porcentagem',
    color='Churn Label',
    text_auto='.1f',
    title='<b>Taxa de Cancelamento (%) por Tipo de Plano</b>',
    color_discrete_map={'No': '#2ecc71', 'Yes': '#e74c3c'},
    barmode='stack'
)
fig_contract.update_layout(
    template='plotly_white',
    yaxis_ticksuffix="%",
    yaxis_range=[0, 100],
    xaxis_title="Tipo de Contrato",
    yaxis_title="Porcentagem (%)"
)
fig_contract.show()

ANÁLISE DE PRODUTOS E FINANCEIRO

In [ ]:
internet_df = df.groupby(['Internet Service', 'Churn Label']).size().reset_index(name='Quantidade')

fig_internet = px.bar(
    internet_df,
    x='Internet Service',
    y='Quantidade',
    color='Churn Label',
    barmode='group',
    text='Quantidade',
    title='<b>Cancelamento por Tipo de Serviço de Internet</b><br><sup>Fibra Óptica é o produto com maior volume absoluto de saídas</sup>',
    labels={'Internet Service': 'Serviço de Internet', 'Quantidade': 'Nº de Clientes', 'Churn Label': 'Cancelou?'},
    color_discrete_map={'No': '#2b5c8f', 'Yes': '#d9534f'}
)
fig_internet.update_traces(textposition='outside')
fig_internet.update_layout(
    template='plotly_white',
    legend_title_text='Status',
    yaxis_range=[0, 2200]
)
fig_internet.show()


pm_df = df.groupby(['Payment Method', 'Churn Label']).size().reset_index(name='Quantidade')

fig_payment = px.bar(
    pm_df,
    x='Payment Method',
    y='Quantidade',
    color='Churn Label',
    barmode='group',
    text='Quantidade',
    title='<b>Formas de Pagamento: Clientes Ativos vs. Cancelados</b><br><sup>Concentração de Churn no Cheque Eletrônico</sup>',
    labels={'Payment Method': 'Forma de Pagamento', 'Quantidade': 'Nº de Clientes', 'Churn Label': 'Cancelou?'},
    color_discrete_map={'No': '#1f77b4', 'Yes': '#ff7f0e'}
)
fig_payment.update_traces(textposition='outside')
fig_payment.update_layout(
    template='plotly_white',
    legend_title_text='Status',
    yaxis_range=[0, 1600]
)
fig_payment.show()

MOTIVOS DE CANCELAMENTO

In [ ]:
churned_df = df[df['Churn Label'] == 'Yes'].dropna(subset=['Churn Reason'])

# Todos os Motivos
reasons_all = (
    churned_df['Churn Reason']
    .value_counts()
    .reset_index()
    .rename(columns={'count': 'Quantidade', 'Churn Reason': 'Motivo'})
    .sort_values(by='Quantidade', ascending=True)
)

fig_reasons_all = px.bar(
    reasons_all,
    x='Quantidade',
    y='Motivo',
    orientation='h',
    text='Quantidade',
    title='<b>Visão Geral: Todos os Motivos de Cancelamento</b>',
    color='Quantidade',
    color_continuous_scale='Reds'
)
fig_reasons_all.update_traces(textposition='outside')
fig_reasons_all.update_layout(
    template='plotly_white',
    height=750,
    coloraxis_showscale=False,
    xaxis_range=[0, 220]
)
fig_reasons_all.show()

# Top 5 Motivos
reasons_top5 = churned_df['Churn Reason'].value_counts().head(5).reset_index()
reasons_top5.columns = ['Motivo', 'Quantidade']
reasons_top5['Porcentagem'] = (reasons_top5['Quantidade'] / len(churned_df) * 100).round(1)
reasons_top5['Label'] = reasons_top5['Quantidade'].astype(str) + " (" + reasons_top5['Porcentagem'].astype(str) + "%)"
reasons_top5 = reasons_top5.sort_values(by='Quantidade', ascending=True)

fig_reasons_top5 = px.bar(
    reasons_top5,
    x='Quantidade',
    y='Motivo',
    orientation='h',
    text='Label',
    title=' <b>OS 5 MAIORES MOTIVOS DE CANCELAMENTO (Foco Estratégico)</b>',
    color='Quantidade',
    color_continuous_scale='OrRd'
)
fig_reasons_top5.update_traces(textposition='outside')
fig_reasons_top5.update_layout(
    template='plotly_white',
    height=400,
    coloraxis_showscale=False,
    xaxis_range=[0, 240]
)
fig_reasons_top5.show()